> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Future-Compatible Retrieval
## Candidate-Prior Deconfounding Diagnostic

This experiment tests whether raw future-distance supervision mixes two distinct effects:

1. **Query-specific compatibility:** candidate \(i\) is unusually suitable for the current query \(q\).
2. **Candidate-global utility / centrality:** candidate \(i\)'s future is broadly close to many query futures, making it a generally safe candidate.

The raw future distance is

\[
d_{qi}=\frac{1}{H}\|\mathbf y_q-\mathbf y_i\|_2^2.
\]

# 1. Candidate prior

Within the same channel, define

\[
b_i=\mathbb E_{q'\sim\mathcal Q_{\mathrm{train},c(i)}}[d(q',i)].
\]

Small \(b_i\) indicates a historical candidate whose future is broadly central/safe, independent of the current query.

# 2. Deconfounded compatibility

For the diagnostic deconfounded target,

\[
\boxed{\tilde d_{qi}=d_{qi}-b_i}.
\]

The listwise target is constructed from the standardized \(\tilde d_{qi}\) with the same temperature as the raw model.

# 3. Leakage-safe construction

During Phase A, the candidate prior is estimated only from training-query futures; validation is used only for model selection. During Phase B, the final refit estimates the prior from train+validation query futures. Test futures are never used for training or selection. Historical candidate futures are used only to define the diagnostic prior/target and are never input to the proposed inference-time scoring function.

# 4. Frozen setting

- Candidate pool: Same-channel
- Target scale: Fixed train-scale
- \(M=100\), \(K=10\)
- Architecture unchanged
- \(\tau_y=0.5\)
- Datasets: Weather and ETTh1
- Horizons: \(H\in\{24,48,96\}\)
- Seeds: \(\{0,1,2\}\)

# 5. Methods

- **Pattern:** cosine/Pattern Top-K.
- **CandidatePrior Diagnostic:** ranks by \(s_{\mathrm{prior}}(i)=-b_i\). This is a diagnostic, not the proposed deployable retriever, because it uses historical candidate futures.
- **Raw Learned / Raw Shuffled:** the Same-Fixed models from the preceding mechanism study.
- **Deconfounded Learned / Deconfounded Shuffled:** trained with \(d_{qi}-b_i\); the shuffled control preserves channel-wise future marginals.

# 6. Questions tested

- Does a strong candidate prior explain the Weather behavior?
- After subtracting the candidate prior, does correct query--future correspondence still beat shuffled correspondence?
- Does the deconfounded model preserve absolute retrieval quality relative to Pattern?

This notebook is used as an **identification diagnostic**, not as the final proposed objective.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Configuration

In [ ]:

from pathlib import Path
import copy
import gc
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_PATHS = {
    "ETTh1":
        REPO_DATA_ROOT / "ETT-small/ETTh1.csv",

    "Weather":
        REPO_DATA_ROOT / "weather/weather.csv",
}

CLEAN_DIR = REPO_WORK_ROOT / "cross_domain_clean"

CLEAN_CACHE = (
    CLEAN_DIR /
    "cache"
)

MECH_DIR = REPO_WORK_ROOT / "semantic_compatibility"

MECH_CACHE = (
    MECH_DIR /
    "cache"
)

MECH_MODEL_DIR = (
    MECH_DIR /
    "models"
)

RESULT_DIR = REPO_WORK_ROOT / "candidate_prior_deconfounding"

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

for p in [
    RESULT_DIR,
    MODEL_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

DATASET_NAMES = [
    "ETTh1",
    "Weather",
]

HORIZONS = [
    24,
    48,
    96,
]

SEQ_LEN = 96

TOP_M = 100
TOP_K = 10

TAU_Y = 0.50

SEEDS = [
    0,
    1,
    2,
]

TRAIN_BATCH = 128
EVAL_BATCH = 256

MAX_MEMORY_WINDOWS = 50000
MAX_TRAIN_QUERIES = 6000
MAX_VAL_QUERIES = 6000
MAX_TEST_QUERIES = 8000

MAX_EPOCHS = 30
PATIENCE = 6

LR = 1e-3
WEIGHT_DECAY = 1e-4

BLOCK_ANCHORS = 10
N_BOOT = 5000

EPS = 1e-8

DATASET_SEED = {
    "ETTh1": 1101,
    "Weather": 2202,
}

FORCE_RETRAIN = False

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )

    print(
        "GPU memory GB:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory /
            1024**3,
            1,
        )
    )

print(
    "Clean experiment:",
    CLEAN_DIR
)

print(
    "Mechanism experiment:",
    MECH_DIR
)

print(
    "Output:",
    RESULT_DIR
)


## 1. Verify prerequisites

In [ ]:

assert CLEAN_DIR.exists()
assert MECH_DIR.exists()

for dataset_name in DATASET_NAMES:

    assert DATA_PATHS[
        dataset_name
    ].exists()

    for H in HORIZONS:

        assert (
            CLEAN_CACHE /
            f"{dataset_name}_H{H}_windows.npz"
        ).exists()

        assert (
            CLEAN_CACHE /
            f"{dataset_name}_H{H}_meta.parquet"
        ).exists()

        assert (
            MECH_CACHE /
            f"{dataset_name}_H{H}_same_channel_topM.npz"
        ).exists()

        for seed in SEEDS:

            assert (
                MECH_MODEL_DIR /
                f"{dataset_name}_H{H}_Same_Fixed_Learned_seed{seed}.pt"
            ).exists()

            assert (
                MECH_MODEL_DIR /
                f"{dataset_name}_H{H}_Same_Fixed_Shuffled_seed{seed}.pt"
            ).exists()

print(
    "All required Same-Fixed caches and raw checkpoints found."
)


## 2. Load raw data and reconstruct train normalization

In [ ]:

def load_numeric_csv(
    path,
):
    df = pd.read_csv(
        path
    )

    timestamp_cols = [
        c
        for c in df.columns
        if str(
            c
        ).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c
        for c in x.columns
        if x[
            c
        ].notna().mean() >
        0.99
    ]

    x = (
        x[
            good_cols
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert np.isfinite(
        x.to_numpy(
            dtype=np.float32
        )
    ).all()

    return x


def split_boundaries(
    dataset_name,
    n,
):
    if dataset_name == "ETTh1":

        train_end = (
            12 *
            30 *
            24
        )

        val_end = (
            train_end +
            4 *
            30 *
            24
        )

    else:

        train_end = int(
            0.70 *
            n
        )

        val_end = int(
            0.80 *
            n
        )

    return {
        "train_end":
            train_end,

        "val_end":
            val_end,

        "test_end":
            n,

        "inner_memory_end":
            int(
                0.60 *
                train_end
            ),
    }


RAW = {}
SPLITS = {}
CHANNEL_NORMALIZED = {}

for dataset_name in DATASET_NAMES:

    df = load_numeric_csv(
        DATA_PATHS[
            dataset_name
        ]
    )

    RAW[
        dataset_name
    ] = df

    s = split_boundaries(
        dataset_name,
        len(
            df
        ),
    )

    SPLITS[
        dataset_name
    ] = s

    train = df.iloc[
        :s[
            "train_end"
        ]
    ]

    mean = train.mean(
        axis=0
    ).to_numpy(
        dtype=np.float32
    )

    std = train.std(
        axis=0,
        ddof=0,
    ).to_numpy(
        dtype=np.float32
    )

    std = np.where(
        std <
        1e-6,
        1.0,
        std,
    ).astype(
        np.float32
    )

    arr = df.to_numpy(
        dtype=np.float32
    )

    CHANNEL_NORMALIZED[
        dataset_name
    ] = (
        (
            arr -
            mean[
                None,
                :
            ]
        ) /
        std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    print(
        dataset_name,
        df.shape,
        s
    )


## 3. Load windows and Same-channel Top-M pools

In [ ]:

WINDOWS = {}
SAME_PRESELECT = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        z = np.load(
            CLEAN_CACHE /
            f"{dataset_name}_H{H}_windows.npz"
        )

        WINDOWS[
            key
        ] = {
            "meta":
                pd.read_parquet(
                    CLEAN_CACHE /
                    f"{dataset_name}_H{H}_meta.parquet"
                ),

            "pattern":
                z[
                    "pattern"
                ].astype(
                    np.float32
                ),

            "context":
                z[
                    "context"
                ].astype(
                    np.float32
                ),

            "local_future":
                z[
                    "future"
                ].astype(
                    np.float32
                ),
        }

        p = np.load(
            MECH_CACHE /
            f"{dataset_name}_H{H}_same_channel_topM.npz"
        )

        SAME_PRESELECT[
            key
        ] = {
            k:
                p[
                    k
                ]
            for k in p.files
        }

print(
    "Loaded all Same-channel pools."
)


## 4. Reconstruct exact deterministic sampled indices

In [ ]:

def full_phase_indices(
    dataset_name,
    H,
):
    meta = WINDOWS[
        (
            dataset_name,
            H
        )
    ][
        "meta"
    ]

    s = SPLITS[
        dataset_name
    ]

    anchor = meta[
        "Anchor"
    ].to_numpy()

    future_end = meta[
        "FutureEnd"
    ].to_numpy()

    return {
        "train_memory":
            np.where(
                future_end <
                s[
                    "inner_memory_end"
                ]
            )[0],

        "train_query":
            np.where(
                (
                    anchor >=
                    s[
                        "inner_memory_end"
                    ]
                )
                &
                (
                    future_end <
                    s[
                        "train_end"
                    ]
                )
            )[0],

        "val_memory":
            np.where(
                future_end <
                s[
                    "train_end"
                ]
            )[0],

        "val_query":
            np.where(
                (
                    anchor >=
                    s[
                        "train_end"
                    ]
                )
                &
                (
                    future_end <
                    s[
                        "val_end"
                    ]
                )
            )[0],

        "test_memory":
            np.where(
                future_end <
                s[
                    "val_end"
                ]
            )[0],

        "test_query":
            np.where(
                anchor >=
                s[
                    "val_end"
                ]
            )[0],
    }


def deterministic_subset(
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        indices
    ) <= max_n:

        return np.sort(
            indices
        )

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            indices,
            size=max_n,
            replace=False,
        )
    )


TASK_INDICES = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        p = full_phase_indices(
            dataset_name,
            H,
        )

        base = (
            DATASET_SEED[
                dataset_name
            ] +
            H *
            10
        )

        sampled = {
            "train_memory":
                deterministic_subset(
                    p[
                        "train_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    1,
                ),

            "train_query":
                deterministic_subset(
                    p[
                        "train_query"
                    ],
                    MAX_TRAIN_QUERIES,
                    base +
                    2,
                ),

            "val_memory":
                deterministic_subset(
                    p[
                        "val_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    3,
                ),

            "val_query":
                deterministic_subset(
                    p[
                        "val_query"
                    ],
                    MAX_VAL_QUERIES,
                    base +
                    4,
                ),

            "test_memory":
                deterministic_subset(
                    p[
                        "test_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    5,
                ),

            "test_query":
                deterministic_subset(
                    p[
                        "test_query"
                    ],
                    MAX_TEST_QUERIES,
                    base +
                    6,
                ),
        }

        TASK_INDICES[
            key
        ] = sampled

        pool = SAME_PRESELECT[
            key
        ]

        for phase in [
            "train",
            "val",
            "test",
        ]:

            np.testing.assert_array_equal(
                pool[
                    f"{phase}_query"
                ],
                sampled[
                    f"{phase}_query"
                ],
            )

print(
    "Exact query sampling reconstruction passed."
)


## 5. Reconstruct Fixed train-scale future targets

In [ ]:

FIXED_FUTURE = {}

for dataset_name in DATASET_NAMES:

    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        meta = WINDOWS[
            key
        ][
            "meta"
        ]

        fixed = np.empty(
            (
                len(
                    meta
                ),
                H,
            ),
            dtype=np.float32,
        )

        local_std = np.empty(
            len(
                meta
            ),
            dtype=np.float32,
        )

        for j, row in enumerate(
            meta.itertuples(
                index=False
            )
        ):

            cidx = int(
                row.ChannelIndex
            )

            anchor = int(
                row.Anchor
            )

            series = arr[
                :,
                cidx
            ]

            past = series[
                anchor -
                SEQ_LEN:
                anchor
            ]

            future = series[
                anchor:
                anchor +
                H
            ]

            s = float(
                np.std(
                    past
                )
            )

            local_std[
                j
            ] = s

            fixed[
                j
            ] = (
                future -
                past[
                    -1
                ]
            )

        reconstructed_local = (
            fixed /
            (
                local_std[
                    :,
                    None
                ] +
                EPS
            )
        )

        max_diff = float(
            np.max(
                np.abs(
                    reconstructed_local -
                    WINDOWS[
                        key
                    ][
                        "local_future"
                    ]
                )
            )
        )

        assert max_diff < 1e-3, (
            dataset_name,
            H,
            max_diff,
        )

        FIXED_FUTURE[
            key
        ] = fixed

        print(
            dataset_name,
            "H=",
            H,
            "| fixed target reconstruction verified"
        )


## 6. Reconstruct observable context scaling

In [ ]:

def fit_robust_scaler(
    x,
):
    med = np.median(
        x,
        axis=0,
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75 -
        q25
    )

    iqr = np.where(
        iqr <
        1e-5,
        1.0,
        iqr,
    )

    return (
        med.astype(
            np.float32
        ),
        iqr.astype(
            np.float32
        ),
    )


def apply_robust_scaler(
    x,
    med,
    iqr,
):
    z = (
        x -
        med
    ) / iqr

    z = np.clip(
        z,
        -8.0,
        8.0,
    )

    return z.astype(
        np.float32
    )


CONTEXT_SCALED = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        train_mem = TASK_INDICES[
            key
        ][
            "train_memory"
        ]

        med, iqr = fit_robust_scaler(
            WINDOWS[
                key
            ][
                "context"
            ][
                train_mem
            ]
        )

        CONTEXT_SCALED[
            key
        ] = apply_robust_scaler(
            WINDOWS[
                key
            ][
                "context"
            ],
            med,
            iqr,
        )


## 7. Candidate-prior statistics

In [ ]:

def build_prior_stats(
    dataset_name,
    H,
    reference_query_indices,
):
    """
    For each channel c, estimate:
      mu_c[h] = E_q[y_q[h]]
      e2_c    = E_q[mean_h y_q[h]^2]

    Then:
      E_q MSE(y_q, y_i)
      = e2_c + mean_h(y_i^2) - 2 mean_h(mu_c * y_i)
    """
    key = (
        dataset_name,
        H
    )

    meta = WINDOWS[
        key
    ][
        "meta"
    ]

    future = FIXED_FUTURE[
        key
    ]

    ref = np.asarray(
        reference_query_indices,
        dtype=np.int64,
    )

    ref_channel = meta.iloc[
        ref
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    stats = {}

    for c in np.unique(
        ref_channel
    ):

        idx = ref[
            ref_channel ==
            c
        ]

        y = future[
            idx
        ]

        assert len(
            y
        ) > 0

        stats[
            int(
                c
            )
        ] = {
            "mean":
                y.mean(
                    axis=0
                ).astype(
                    np.float32
                ),

            "mean_sq":
                float(
                    np.mean(
                        y ** 2
                    )
                ),

            "count":
                int(
                    len(
                        y
                    )
                ),
        }

    return stats


PRIOR_STATS_PHASE_A = {}
PRIOR_STATS_REFIT = {}

prior_stats_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        train_q = TASK_INDICES[
            key
        ][
            "train_query"
        ]

        val_q = TASK_INDICES[
            key
        ][
            "val_query"
        ]

        refit_q = np.concatenate(
            [
                train_q,
                val_q,
            ]
        )

        a = build_prior_stats(
            dataset_name,
            H,
            train_q,
        )

        b = build_prior_stats(
            dataset_name,
            H,
            refit_q,
        )

        PRIOR_STATS_PHASE_A[
            key
        ] = a

        PRIOR_STATS_REFIT[
            key
        ] = b

        for label, stats in [
            (
                "PhaseA_TrainOnly",
                a,
            ),
            (
                "Refit_TrainPlusVal",
                b,
            ),
        ]:

            for c, s in stats.items():

                prior_stats_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "PriorPhase":
                        label,

                    "ChannelIndex":
                        c,

                    "ReferenceCount":
                        s[
                            "count"
                        ],

                    "MeanFutureL2":
                        float(
                            np.sqrt(
                                np.mean(
                                    s[
                                        "mean"
                                    ] ** 2
                                )
                            )
                        ),

                    "MeanFutureEnergy":
                        s[
                            "mean_sq"
                        ],
                })

prior_stats_table = pd.DataFrame(
    prior_stats_rows
)

display(
    prior_stats_table.head(
        20
    )
)

prior_stats_table.to_csv(
    RESULT_DIR /
    "01_candidate_prior_reference_summary.csv",
    index=False,
)


## 8. Compute candidate-prior matrix

In [ ]:

def candidate_prior_matrix(
    dataset_name,
    H,
    cand_idx,
    q_channel,
    prior_stats,
):
    key = (
        dataset_name,
        H
    )

    future = FIXED_FUTURE[
        key
    ]

    cand_future = future[
        cand_idx
    ]

    B, M, HH = cand_future.shape

    out = np.empty(
        (
            B,
            M,
        ),
        dtype=np.float32,
    )

    for c in np.unique(
        q_channel
    ):

        pos = np.where(
            q_channel ==
            c
        )[0]

        assert int(
            c
        ) in prior_stats

        mu = prior_stats[
            int(
                c
            )
        ][
            "mean"
        ]

        mean_sq = prior_stats[
            int(
                c
            )
        ][
            "mean_sq"
        ]

        yi = cand_future[
            pos
        ]

        yi_sq = np.mean(
            yi ** 2,
            axis=2,
        )

        cross = np.mean(
            yi *
            mu[
                None,
                None,
                :
            ],
            axis=2,
        )

        prior = (
            mean_sq +
            yi_sq -
            2.0 *
            cross
        )

        out[
            pos
        ] = prior.astype(
            np.float32
        )

    assert np.isfinite(
        out
    ).all()

    return out


## 9. Phase packaging

In [ ]:

def phase_data(
    dataset_name,
    H,
    phase,
    prior_stats,
):
    key = (
        dataset_name,
        H
    )

    w = WINDOWS[
        key
    ]

    p = SAME_PRESELECT[
        key
    ]

    q_idx = p[
        f"{phase}_query"
    ]

    cand_idx = p[
        f"{phase}_idx"
    ]

    meta = w[
        "meta"
    ]

    q_channel = meta.iloc[
        q_idx
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    cand_channel = meta.iloc[
        cand_idx.reshape(
            -1
        )
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    ).reshape(
        cand_idx.shape
    )

    # Same-channel pool must be exact.
    assert np.all(
        cand_channel ==
        q_channel[
            :,
            None
        ]
    )

    prior = candidate_prior_matrix(
        dataset_name,
        H,
        cand_idx,
        q_channel,
        prior_stats,
    )

    return {
        "q_idx":
            q_idx,

        "cand_idx":
            cand_idx,

        "pattern_score":
            p[
                f"{phase}_score"
            ].astype(
                np.float32
            ),

        "q_context":
            CONTEXT_SCALED[
                key
            ][
                q_idx
            ],

        "cand_context":
            CONTEXT_SCALED[
                key
            ][
                cand_idx
            ],

        "q_future":
            FIXED_FUTURE[
                key
            ][
                q_idx
            ],

        "cand_future":
            FIXED_FUTURE[
                key
            ][
                cand_idx
            ],

        "candidate_prior":
            prior,

        "q_anchor":
            meta.iloc[
                q_idx
            ][
                "Anchor"
            ].to_numpy(
                dtype=np.int64
            ),

        "q_channel":
            q_channel,
    }


## 10. Candidate-prior diagnostic

In [ ]:

prior_diag_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
            PRIOR_STATS_REFIT[
                key
            ],
        )

        prior = test_phase[
            "candidate_prior"
        ]

        prior_diag_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "CandidatePrior_Mean":
                float(
                    prior.mean()
                ),

            "CandidatePrior_Std":
                float(
                    prior.std()
                ),

            "CandidatePrior_Q01":
                float(
                    np.quantile(
                        prior,
                        0.01,
                    )
                ),

            "CandidatePrior_Q50":
                float(
                    np.quantile(
                        prior,
                        0.50,
                    )
                ),

            "CandidatePrior_Q99":
                float(
                    np.quantile(
                        prior,
                        0.99,
                    )
                ),
        })

prior_diag_table = pd.DataFrame(
    prior_diag_rows
)

display(
    prior_diag_table
)

prior_diag_table.to_csv(
    RESULT_DIR /
    "02_candidate_prior_distribution.csv",
    index=False,
)


## 11. Retrieval metrics

In [ ]:

def future_distance(
    q_future,
    cand_future,
):
    return np.mean(
        (
            cand_future -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    )


def topk_from_scores(
    score,
    k,
):
    idx = np.argpartition(
        -score,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    row = np.arange(
        len(
            score
        )
    )[
        :,
        None
    ]

    local_score = score[
        row,
        idx
    ]

    order = np.argsort(
        -local_score,
        axis=1,
    )

    return idx[
        row,
        order
    ]


def gather2(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx
    ]


def gather3(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx,
        :
    ]


def ndcg_at_k(
    score,
    fdist,
    k,
):
    mean = fdist.mean(
        axis=1,
        keepdims=True,
    )

    std = (
        fdist.std(
            axis=1,
            keepdims=True,
        ) +
        1e-6
    )

    z = (
        fdist -
        mean
    ) / std

    relevance = np.exp(
        -z /
        TAU_Y
    )

    selected = topk_from_scores(
        score,
        k,
    )

    ideal = np.argsort(
        -relevance,
        axis=1,
    )[
        :,
        :k
    ]

    rel_sel = gather2(
        relevance,
        selected,
    )

    rel_ideal = gather2(
        relevance,
        ideal,
    )

    discount = (
        1.0 /
        np.log2(
            np.arange(
                2,
                k +
                2
            )
        )
    )[
        None,
        :
    ]

    dcg = np.sum(
        rel_sel *
        discount,
        axis=1,
    )

    idcg = (
        np.sum(
            rel_ideal *
            discount,
            axis=1,
        ) +
        EPS
    )

    return (
        dcg /
        idcg
    ).astype(
        np.float32
    )


def oracle_recall_at_k(
    selected,
    fdist,
    k,
):
    oracle = np.argpartition(
        fdist,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    out = np.empty(
        len(
            selected
        ),
        dtype=np.float32,
    )

    for i in range(
        len(
            selected
        )
    ):

        out[
            i
        ] = (
            len(
                set(
                    selected[
                        i
                    ].tolist()
                )
                &
                set(
                    oracle[
                        i
                    ].tolist()
                )
            )
            /
            k
        )

    return out


def query_metrics(
    score,
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    selected = topk_from_scores(
        score,
        TOP_K,
    )

    selected_dist = gather2(
        fdist,
        selected,
    )

    selected_future = gather3(
        phase[
            "cand_future"
        ],
        selected,
    )

    pred = selected_future.mean(
        axis=1
    )

    forecast_mse = np.mean(
        (
            pred -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    return pd.DataFrame({
        "AnalogFutureMSE":
            selected_dist.mean(
                axis=1
            ).astype(
                np.float32
            ),

        "RetrievalForecastMSE":
            forecast_mse.astype(
                np.float32
            ),

        "NDCG@K":
            ndcg_at_k(
                score,
                fdist,
                TOP_K,
            ),

        "OracleRecall@K":
            oracle_recall_at_k(
                selected,
                fdist,
                TOP_K,
            ),
    })


## 12. Future-Compatible reranker

In [ ]:

class FutureCompatibleReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim=7,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim //
                2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim //
                2,
                1,
            ),
        )

        raw_alpha = math.log(
            math.exp(
                initial_alpha
            ) -
            1.0
        )

        self.raw_alpha = nn.Parameter(
            torch.tensor(
                raw_alpha,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.raw_alpha
        )

    def forward(
        self,
        pattern_score,
        q_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            q_context[
                :,
                None,
                :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ...,
                    None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(
                -1
            )
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )


## 13. Deconfounded listwise loss

In [ ]:

def deconfounded_listwise_loss(
    score,
    future_dist,
    candidate_prior,
):
    residual_dist = (
        future_dist -
        candidate_prior
    )

    mean = residual_dist.mean(
        dim=1,
        keepdim=True,
    )

    std = residual_dist.std(
        dim=1,
        keepdim=True,
        unbiased=False,
    ).clamp_min(
        1e-6
    )

    z = (
        residual_dist -
        mean
    ) / std

    target = torch.softmax(
        -z /
        TAU_Y,
        dim=1,
    )

    log_prob = F.log_softmax(
        score,
        dim=1,
    )

    loss = -(
        target *
        log_prob
    ).sum(
        dim=1
    ).mean()

    assert torch.isfinite(
        loss
    )

    return loss


## 14. Training helpers

In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def channelwise_shuffled_future(
    phase,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    out = phase[
        "q_future"
    ].copy()

    channels = phase[
        "q_channel"
    ]

    for c in np.unique(
        channels
    ):

        pos = np.where(
            channels ==
            c
        )[0]

        if len(
            pos
        ) <= 1:
            continue

        perm = rng.permutation(
            pos
        )

        out[
            pos
        ] = phase[
            "q_future"
        ][
            perm
        ]

    return out


def train_epoch(
    model,
    optimizer,
    phase,
    shuffled_future=None,
):
    model.train()

    n = len(
        phase[
            "q_idx"
        ]
    )

    order = np.random.permutation(
        n
    )

    losses = []

    for start in range(
        0,
        n,
        TRAIN_BATCH,
    ):

        ids = order[
            start:
            start +
            TRAIN_BATCH
        ]

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cf = torch.tensor(
            phase[
                "cand_future"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        prior = torch.tensor(
            phase[
                "candidate_prior"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qf_np = (
            phase[
                "q_future"
            ][
                ids
            ]
            if shuffled_future is None
            else
            shuffled_future[
                ids
            ]
        )

        qf = torch.tensor(
            qf_np,
            dtype=torch.float32,
            device=DEVICE,
        )

        fdist = (
            (
                cf -
                qf[
                    :,
                    None,
                    :
                ]
            ) ** 2
        ).mean(
            dim=2
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        score = model(
            ps,
            qc,
            cc,
        )

        loss = deconfounded_listwise_loss(
            score,
            fdist,
            prior,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def predict_scores(
    model,
    phase,
):
    model.eval()

    chunks = []

    n = len(
        phase[
            "q_idx"
        ]
    )

    for start in range(
        0,
        n,
        EVAL_BATCH,
    ):

        end = min(
            start +
            EVAL_BATCH,
            n,
        )

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        chunks.append(
            model(
                ps,
                qc,
                cc,
            ).cpu()
        )

    return (
        torch.cat(
            chunks,
            dim=0,
        ).numpy().astype(
            np.float32
        )
    )


## 15. Phase A epoch selection

In [ ]:

def select_epoch(
    dataset_name,
    H,
    seed,
    shuffled=False,
):
    key = (
        dataset_name,
        H
    )

    set_seed(
        seed
    )

    prior_stats = PRIOR_STATS_PHASE_A[
        key
    ]

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        prior_stats,
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        prior_stats,
    )

    model = FutureCompatibleReranker().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None

    if shuffled:

        shuffled_train = channelwise_shuffled_future(
            train_phase,
            seed=(
                100000 +
                seed
            ),
        )

    best_epoch = None
    best_val = float(
        "inf"
    )

    wait = 0
    rows = []

    for epoch in range(
        1,
        MAX_EPOCHS +
        1,
    ):

        loss = train_epoch(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        val_score = predict_scores(
            model,
            val_phase,
        )

        val_m = query_metrics(
            val_score,
            val_phase,
        )

        val_analog = float(
            val_m[
                "AnalogFutureMSE"
            ].mean()
        )

        rows.append({
            "Epoch":
                epoch,

            "TrainLoss":
                loss,

            "ValAnalogFutureMSE":
                val_analog,

            "ValNDCG@K":
                float(
                    val_m[
                        "NDCG@K"
                    ].mean()
                ),

            "Alpha":
                float(
                    model.alpha.item()
                ),
        })

        if (
            val_analog <
            best_val -
            1e-10
        ):

            best_val = val_analog
            best_epoch = epoch
            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    assert best_epoch is not None

    return (
        best_epoch,
        best_val,
        pd.DataFrame(
            rows
        ),
    )


## 16. Phase B refit from scratch

In [ ]:

def refit_model(
    dataset_name,
    H,
    seed,
    epochs,
    shuffled=False,
):
    key = (
        dataset_name,
        H
    )

    set_seed(
        seed
    )

    prior_stats = PRIOR_STATS_REFIT[
        key
    ]

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        prior_stats,
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        prior_stats,
    )

    model = FutureCompatibleReranker().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None
    shuffled_val = None

    if shuffled:

        shuffled_train = channelwise_shuffled_future(
            train_phase,
            seed=(
                200000 +
                seed
            ),
        )

        shuffled_val = channelwise_shuffled_future(
            val_phase,
            seed=(
                300000 +
                seed
            ),
        )

    for _ in range(
        epochs
    ):

        train_epoch(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        train_epoch(
            model,
            optimizer,
            val_phase,
            shuffled_future=shuffled_val,
        )

    return model


## 17. Train/load Deconfounded Learned and Shuffled models

In [ ]:

DECONF_MODELS = {}
training_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for method_name, shuffled in [
            (
                "DeconfLearned",
                False,
            ),
            (
                "DeconfShuffled",
                True,
            ),
        ]:

            for seed in SEEDS:

                path = (
                    MODEL_DIR /
                    f"{dataset_name}_H{H}_{method_name}_seed{seed}.pt"
                )

                if (
                    path.exists()
                    and
                    not
                    FORCE_RETRAIN
                ):

                    ckpt = torch.load(
                        path,
                        map_location="cpu",
                        weights_only=False,
                    )

                    model = FutureCompatibleReranker().to(
                        DEVICE
                    )

                    model.load_state_dict(
                        ckpt[
                            "StateDict"
                        ]
                    )

                    best_epoch = int(
                        ckpt[
                            "BestEpoch"
                        ]
                    )

                    best_val = float(
                        ckpt[
                            "BestValidationAnalogFutureMSE"
                        ]
                    )

                    source = "loaded"

                else:

                    (
                        best_epoch,
                        best_val,
                        history,
                    ) = select_epoch(
                        dataset_name,
                        H,
                        seed,
                        shuffled=shuffled,
                    )

                    model = refit_model(
                        dataset_name,
                        H,
                        seed,
                        best_epoch,
                        shuffled=shuffled,
                    )

                    torch.save(
                        {
                            "Dataset":
                                dataset_name,

                            "Horizon":
                                H,

                            "Method":
                                method_name,

                            "Seed":
                                seed,

                            "BestEpoch":
                                best_epoch,

                            "BestValidationAnalogFutureMSE":
                                best_val,

                            "Target":
                                "FutureDistanceMinusCandidatePrior",

                            "StateDict":
                                model.state_dict(),
                        },
                        path,
                    )

                    source = "trained"

                model.eval()

                DECONF_MODELS[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed
                    )
                ] = model

                training_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Method":
                        method_name,

                    "Seed":
                        seed,

                    "BestEpoch":
                        best_epoch,

                    "BestValidationAnalogFutureMSE":
                        best_val,

                    "Alpha":
                        float(
                            model.alpha.item()
                        ),

                    "Source":
                        source,
                })

                print(
                    dataset_name,
                    "H=",
                    H,
                    method_name,
                    "seed=",
                    seed,
                    "|",
                    source,
                    "| epoch",
                    best_epoch,
                    "| val",
                    best_val,
                )

training_table = pd.DataFrame(
    training_rows
)

training_table.to_csv(
    RESULT_DIR /
    "03_training_summary.csv",
    index=False,
)


## 18. Load previous Raw Same-Fixed models

In [ ]:

RAW_MODELS = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for method_name, file_tag in [
            (
                "RawLearned",
                "Learned",
            ),
            (
                "RawShuffled",
                "Shuffled",
            ),
        ]:

            for seed in SEEDS:

                path = (
                    MECH_MODEL_DIR /
                    f"{dataset_name}_H{H}_Same_Fixed_{file_tag}_seed{seed}.pt"
                )

                ckpt = torch.load(
                    path,
                    map_location="cpu",
                    weights_only=False,
                )

                model = FutureCompatibleReranker().to(
                    DEVICE
                )

                model.load_state_dict(
                    ckpt[
                        "StateDict"
                    ]
                )

                model.eval()

                RAW_MODELS[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed
                    )
                ] = model

print(
    "Loaded all previous Raw Same-Fixed models."
)


## 19. Evaluate all methods

In [ ]:

METHOD_QUERY = {}
seed_rows = []
main_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
            PRIOR_STATS_REFIT[
                key
            ],
        )

        # Pattern
        pattern_score = test_phase[
            "pattern_score"
        ]

        pattern_m = query_metrics(
            pattern_score,
            test_phase,
        )

        # Candidate-prior diagnostic:
        # lower b_i = more generically central/safe candidate
        prior_score = -test_phase[
            "candidate_prior"
        ]

        prior_m = query_metrics(
            prior_score,
            test_phase,
        )

        method_seed_frames = {}

        for method_name, model_store in [
            (
                "RawLearned",
                RAW_MODELS,
            ),
            (
                "RawShuffled",
                RAW_MODELS,
            ),
            (
                "DeconfLearned",
                DECONF_MODELS,
            ),
            (
                "DeconfShuffled",
                DECONF_MODELS,
            ),
        ]:

            frames = []

            for seed in SEEDS:

                model = model_store[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed
                    )
                ]

                score = predict_scores(
                    model,
                    test_phase,
                )

                m = query_metrics(
                    score,
                    test_phase,
                )

                frames.append(
                    m
                )

                seed_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Method":
                        method_name,

                    "Seed":
                        seed,

                    "AnalogFutureMSE":
                        float(
                            m[
                                "AnalogFutureMSE"
                            ].mean()
                        ),

                    "RetrievalForecastMSE":
                        float(
                            m[
                                "RetrievalForecastMSE"
                            ].mean()
                        ),

                    "NDCG@K":
                        float(
                            m[
                                "NDCG@K"
                            ].mean()
                        ),

                    "OracleRecall@K":
                        float(
                            m[
                                "OracleRecall@K"
                            ].mean()
                        ),
                })

            method_seed_frames[
                method_name
            ] = frames

        mean_metrics = {
            "Pattern":
                pattern_m,

            "CandidatePriorDiagnostic":
                prior_m,
        }

        for method_name, frames in method_seed_frames.items():

            mean_metrics[
                method_name
            ] = pd.DataFrame({
                col:
                    np.stack(
                        [
                            f[
                                col
                            ].to_numpy()
                            for f in frames
                        ],
                        axis=0,
                    ).mean(
                        axis=0
                    )

                for col in frames[
                    0
                ].columns
            })

        q = pd.DataFrame({
            "Anchor":
                test_phase[
                    "q_anchor"
                ],

            "ChannelIndex":
                test_phase[
                    "q_channel"
                ],
        })

        for method_name, m in mean_metrics.items():

            for col in m.columns:

                q[
                    f"{method_name}_{col}"
                ] = m[
                    col
                ].to_numpy()

        METHOD_QUERY[
            key
        ] = q

        q.to_parquet(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.parquet",
            index=False,
        )

        for method_name, m in mean_metrics.items():

            row = {
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Method":
                    method_name,

                "AnalogFutureMSE":
                    float(
                        m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "RetrievalForecastMSE":
                    float(
                        m[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "NDCG@K":
                    float(
                        m[
                            "NDCG@K"
                        ].mean()
                    ),

                "OracleRecall@K":
                    float(
                        m[
                            "OracleRecall@K"
                        ].mean()
                    ),
            }

            main_rows.append(
                row
            )

seed_table = pd.DataFrame(
    seed_rows
)

main_table = pd.DataFrame(
    main_rows
)

display(
    main_table.sort_values(
        [
            "Dataset",
            "Horizon",
            "AnalogFutureMSE",
        ]
    )
)

seed_table.to_csv(
    RESULT_DIR /
    "04_seed_results.csv",
    index=False,
)

main_table.to_csv(
    RESULT_DIR /
    "05_main_deconfounding_summary.csv",
    index=False,
)


## 20. Core comparison table

In [ ]:

comparison_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        t = (
            main_table[
                (
                    main_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    main_table[
                        "Horizon"
                    ] ==
                    H
                )
            ]
            .set_index(
                "Method"
            )
        )

        pattern = t.loc[
            "Pattern",
            "AnalogFutureMSE",
        ]

        prior = t.loc[
            "CandidatePriorDiagnostic",
            "AnalogFutureMSE",
        ]

        raw = t.loc[
            "RawLearned",
            "AnalogFutureMSE",
        ]

        raw_shuffle = t.loc[
            "RawShuffled",
            "AnalogFutureMSE",
        ]

        deconf = t.loc[
            "DeconfLearned",
            "AnalogFutureMSE",
        ]

        deconf_shuffle = t.loc[
            "DeconfShuffled",
            "AnalogFutureMSE",
        ]

        comparison_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Pattern":
                pattern,

            "CandidatePriorDiagnostic":
                prior,

            "RawLearned":
                raw,

            "RawShuffled":
                raw_shuffle,

            "DeconfLearned":
                deconf,

            "DeconfShuffled":
                deconf_shuffle,

            "RawLearned_vs_Pattern_%":
                100.0 *
                (
                    pattern -
                    raw
                ) /
                pattern,

            "RawLearned_vs_RawShuffled_%":
                100.0 *
                (
                    raw_shuffle -
                    raw
                ) /
                raw_shuffle,

            "DeconfLearned_vs_Pattern_%":
                100.0 *
                (
                    pattern -
                    deconf
                ) /
                pattern,

            "DeconfLearned_vs_DeconfShuffled_%":
                100.0 *
                (
                    deconf_shuffle -
                    deconf
                ) /
                deconf_shuffle,

            "DeconfLearned_vs_RawLearned_%":
                100.0 *
                (
                    raw -
                    deconf
                ) /
                raw,

            "CandidatePrior_vs_Pattern_%":
                100.0 *
                (
                    pattern -
                    prior
                ) /
                pattern,
        })

comparison_table = pd.DataFrame(
    comparison_rows
)

display(
    comparison_table
)

comparison_table.to_csv(
    RESULT_DIR /
    "06_core_comparison.csv",
    index=False,
)


## 21. Reranker-residual correlation with Candidate Prior

In [ ]:

def rowwise_pearson(
    a,
    b,
):
    a = np.asarray(
        a,
        dtype=np.float64,
    )

    b = np.asarray(
        b,
        dtype=np.float64,
    )

    ac = (
        a -
        a.mean(
            axis=1,
            keepdims=True,
        )
    )

    bc = (
        b -
        b.mean(
            axis=1,
            keepdims=True,
        )
    )

    num = np.sum(
        ac *
        bc,
        axis=1,
    )

    den = (
        np.sqrt(
            np.sum(
                ac ** 2,
                axis=1,
            ) *
            np.sum(
                bc ** 2,
                axis=1,
            )
        )
        +
        1e-12
    )

    return (
        num /
        den
    ).astype(
        np.float32
    )


correlation_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
            PRIOR_STATS_REFIT[
                key
            ],
        )

        pattern = test_phase[
            "pattern_score"
        ]

        prior_score = -test_phase[
            "candidate_prior"
        ]

        for method_name, store in [
            (
                "RawLearned",
                RAW_MODELS,
            ),
            (
                "RawShuffled",
                RAW_MODELS,
            ),
            (
                "DeconfLearned",
                DECONF_MODELS,
            ),
            (
                "DeconfShuffled",
                DECONF_MODELS,
            ),
        ]:

            seed_corr = []

            for seed in SEEDS:

                model = store[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed
                    )
                ]

                full_score = predict_scores(
                    model,
                    test_phase,
                )

                # Reranker correction beyond the pattern baseline.
                residual = (
                    full_score -
                    pattern
                )

                corr = rowwise_pearson(
                    residual,
                    prior_score,
                )

                seed_corr.append(
                    corr
                )

                correlation_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Method":
                        method_name,

                    "Seed":
                        seed,

                    "MeanResidualVsNegPriorCorrelation":
                        float(
                            np.nanmean(
                                corr
                            )
                        ),

                    "MedianResidualVsNegPriorCorrelation":
                        float(
                            np.nanmedian(
                                corr
                            )
                        ),
                })

correlation_table = pd.DataFrame(
    correlation_rows
)

display(
    correlation_table
)

correlation_table.to_csv(
    RESULT_DIR /
    "07_score_prior_correlation.csv",
    index=False,
)


## 22. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(
        x
    )

    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        parts = []

        for _ in range(
            n_blocks
        ):

            start = rng.integers(
                0,
                max_start +
                1,
            )

            parts.append(
                x[
                    start:
                    start +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),
    }


bootstrap_rows = []

comparisons = [
    (
        "Pattern",
        "DeconfLearned",
    ),
    (
        "DeconfShuffled",
        "DeconfLearned",
    ),
    (
        "RawLearned",
        "DeconfLearned",
    ),
    (
        "RawShuffled",
        "RawLearned",
    ),
]

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        q = METHOD_QUERY[
            key
        ]

        for baseline, proposed in comparisons:

            for metric in [
                "AnalogFutureMSE",
                "RetrievalForecastMSE",
            ]:

                tmp = pd.DataFrame({
                    "Anchor":
                        q[
                            "Anchor"
                        ],

                    "Diff":
                        (
                            q[
                                f"{baseline}_{metric}"
                            ]
                            -
                            q[
                                f"{proposed}_{metric}"
                            ]
                        ),
                })

                anchor_diff = (
                    tmp
                    .groupby(
                        "Anchor"
                    )[
                        "Diff"
                    ]
                    .mean()
                    .sort_index()
                    .to_numpy()
                )

                seed = (
                    DATASET_SEED[
                        dataset_name
                    ]
                    +
                    H *
                    100
                    +
                    len(
                        bootstrap_rows
                    )
                )

                r = moving_block_bootstrap(
                    anchor_diff,
                    BLOCK_ANCHORS,
                    N_BOOT,
                    seed,
                )

                r.update({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Baseline":
                        baseline,

                    "Proposed":
                        proposed,

                    "Metric":
                        metric,

                    "SignificantImprovement":
                        bool(
                            r[
                                "CI_2.5%"
                            ] >
                            0
                        ),
                })

                bootstrap_rows.append(
                    r
                )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "08_moving_block_bootstrap.csv",
    index=False,
)


## 23. Per-channel Weather analysis

In [ ]:

weather_rows = []

for H in HORIZONS:

    key = (
        "Weather",
        H
    )

    q = METHOD_QUERY[
        key
    ]

    meta = WINDOWS[
        key
    ][
        "meta"
    ]

    channel_map = (
        meta[
            [
                "ChannelIndex",
                "Channel",
            ]
        ]
        .drop_duplicates()
        .set_index(
            "ChannelIndex"
        )[
            "Channel"
        ]
        .to_dict()
    )

    for cidx, g in q.groupby(
        "ChannelIndex"
    ):

        pattern = float(
            g[
                "Pattern_AnalogFutureMSE"
            ].mean()
        )

        raw = float(
            g[
                "RawLearned_AnalogFutureMSE"
            ].mean()
        )

        deconf = float(
            g[
                "DeconfLearned_AnalogFutureMSE"
            ].mean()
        )

        deconf_shuffle = float(
            g[
                "DeconfShuffled_AnalogFutureMSE"
            ].mean()
        )

        weather_rows.append({
            "Horizon":
                H,

            "ChannelIndex":
                int(
                    cidx
                ),

            "Channel":
                str(
                    channel_map[
                        int(
                            cidx
                        )
                    ]
                ),

            "N":
                len(
                    g
                ),

            "Pattern":
                pattern,

            "RawLearned":
                raw,

            "DeconfLearned":
                deconf,

            "DeconfShuffled":
                deconf_shuffle,

            "RawLearnedVsPattern_%":
                100.0 *
                (
                    pattern -
                    raw
                ) /
                pattern,

            "DeconfLearnedVsPattern_%":
                100.0 *
                (
                    pattern -
                    deconf
                ) /
                pattern,

            "DeconfLearnedVsShuffled_%":
                100.0 *
                (
                    deconf_shuffle -
                    deconf
                ) /
                deconf_shuffle,
        })

weather_table = pd.DataFrame(
    weather_rows
)

display(
    weather_table
)

weather_table.to_csv(
    RESULT_DIR /
    "09_weather_per_channel.csv",
    index=False,
)


## 24. Final deconfounding decision

In [ ]:

def row_for(
    dataset_name,
    H,
):
    return comparison_table[
        (
            comparison_table[
                "Dataset"
            ] ==
            dataset_name
        )
        &
        (
            comparison_table[
                "Horizon"
            ] ==
            H
        )
    ].iloc[
        0
    ]


weather_vs_pattern = 0
weather_vs_shuffle = 0
weather_vs_raw = 0

etth1_vs_pattern = 0
etth1_vs_shuffle = 0

for H in HORIZONS:

    w = row_for(
        "Weather",
        H,
    )

    e = row_for(
        "ETTh1",
        H,
    )

    if (
        w[
            "DeconfLearned"
        ] <
        w[
            "Pattern"
        ]
    ):
        weather_vs_pattern += 1

    if (
        w[
            "DeconfLearned"
        ] <
        w[
            "DeconfShuffled"
        ]
    ):
        weather_vs_shuffle += 1

    if (
        w[
            "DeconfLearned"
        ] <
        w[
            "RawLearned"
        ]
    ):
        weather_vs_raw += 1

    if (
        e[
            "DeconfLearned"
        ] <
        e[
            "Pattern"
        ]
    ):
        etth1_vs_pattern += 1

    if (
        e[
            "DeconfLearned"
        ] <
        e[
            "DeconfShuffled"
        ]
    ):
        etth1_vs_shuffle += 1


weather_boot_shuffle = bootstrap_table[
    (
        bootstrap_table[
            "Dataset"
        ] ==
        "Weather"
    )
    &
    (
        bootstrap_table[
            "Baseline"
        ] ==
        "DeconfShuffled"
    )
    &
    (
        bootstrap_table[
            "Proposed"
        ] ==
        "DeconfLearned"
    )
    &
    (
        bootstrap_table[
            "Metric"
        ] ==
        "AnalogFutureMSE"
    )
]

weather_sig_vs_shuffle = int(
    weather_boot_shuffle[
        "SignificantImprovement"
    ].sum()
)


raw_shuffle_corr_weather = float(
    correlation_table[
        (
            correlation_table[
                "Dataset"
            ] ==
            "Weather"
        )
        &
        (
            correlation_table[
                "Method"
            ] ==
            "RawShuffled"
        )
    ][
        "MeanResidualVsNegPriorCorrelation"
    ].mean()
)

deconf_shuffle_corr_weather = float(
    correlation_table[
        (
            correlation_table[
                "Dataset"
            ] ==
            "Weather"
        )
        &
        (
            correlation_table[
                "Method"
            ] ==
            "DeconfShuffled"
        )
    ][
        "MeanResidualVsNegPriorCorrelation"
    ].mean()
)


if (
    weather_vs_pattern == 3
    and
    weather_vs_shuffle == 3
    and
    etth1_vs_pattern == 3
    and
    etth1_vs_shuffle == 3
):

    interpretation = (
        "Candidate-prior deconfounding resolves the key supervision confound. "
        "Future-compatible relevance remains predictive while within-channel shuffled supervision fails."
    )

    next_step = (
        "Stop model development. Promote deconfounded future compatibility to the final method, "
        "run the final cross-domain replication with this target, then move to theory and manuscript."
    )

elif (
    weather_vs_pattern >= 2
    and
    weather_vs_shuffle >= 2
    and
    etth1_vs_pattern == 3
):

    interpretation = (
        "Candidate-prior deconfounding substantially improves identifiability of query-specific relevance, "
        "but Weather remains horizon-dependent."
    )

    next_step = (
        "Do not add architecture complexity. Inspect the one failing Weather horizon/channel and "
        "decide whether the final claim should be qualified."
    )

else:

    interpretation = (
        "Subtracting candidate prior does not fully resolve the Weather shuffled-control failure."
    )

    next_step = (
        "Do not force the deconfounded target. Treat Weather as a failure case and center the broad claim "
        "on the domains where correct future supervision is identifiable."
    )


decision = pd.DataFrame(
    [
        {
            "Weather_DeconfBeatsPattern":
                weather_vs_pattern,

            "Weather_DeconfBeatsDeconfShuffled":
                weather_vs_shuffle,

            "Weather_SignificantVsDeconfShuffled":
                weather_sig_vs_shuffle,

            "Weather_DeconfBeatsRawLearned":
                weather_vs_raw,

            "ETTh1_DeconfBeatsPattern":
                etth1_vs_pattern,

            "ETTh1_DeconfBeatsDeconfShuffled":
                etth1_vs_shuffle,

            "Weather_RawShuffledResidualVsNegPriorCorr":
                raw_shuffle_corr_weather,

            "Weather_DeconfShuffledResidualVsNegPriorCorr":
                deconf_shuffle_corr_weather,

            "Interpretation":
                interpretation,

            "NextStep":
                next_step,
        }
    ]
)

display(
    decision
)

decision.to_csv(
    RESULT_DIR /
    "10_final_deconfounding_decision.csv",
    index=False,
)


# Result Interpretation

Default output directory:

```text
_work/candidate_prior_deconfounding/
```

Key files:

```text
02_candidate_prior_distribution.csv
05_main_deconfounding_summary.csv
06_core_comparison.csv
07_score_prior_correlation.csv
08_moving_block_bootstrap.csv
09_weather_per_channel.csv
10_final_deconfounding_decision.csv
```

## 1. Candidate Prior diagnostic

If CandidatePrior is substantially better than Pattern on Weather, selecting globally central historical futures alone explains a large part of retrieval quality. Because the diagnostic uses candidate futures, it is **not** claimed as the proposed inference-time method.

## 2. Why a shuffled model can remain strong

`07_score_prior_correlation.csv` measures whether the Shuffled reranker residual correlates with \(-b_i\). A strong positive correlation indicates that the model can learn candidate centrality even after query--future correspondence is destroyed.

## 3. Deconfounded Correct vs. Shuffled

The comparison

\[
\text{DeconfLearned}<\text{DeconfShuffled}
\]

diagnoses residual query-specific signal after removing candidate-global utility.

## 4. Absolute retrieval quality

Deconfounding need not improve absolute AnalogFutureMSE. In the executed experiment it is primarily useful for separating the mechanisms, and the paper therefore retains the simpler raw future-compatible objective as the proposed method.

## 5. ETTh1

ETTh1 serves as a contrasting query-specific dataset and tests whether the mechanism is specific to Weather.
